In [1]:
import sys
sys.path.append('../input/iterative-stratification/iterative-stratification-master')
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

sys.path.append('../input/autograd')
from autograd import grad
import autograd.numpy as anp
from scipy.optimize import fsolve
import datetime

import os
import gc
import math
from time import time
import random

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

from sklearn.decomposition import PCA
from sklearn.preprocessing import QuantileTransformer

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.tensorboard import SummaryWriter       
writer = SummaryWriter('./log')
if not os.path.exists('./log'):
    os.makedirs('log')

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(42)

In [3]:
def load_data(path = '../input/lish-moa/'):
    train_features = pd.read_csv(os.path.join(path, 'train_features.csv'))
    test_features = pd.read_csv(os.path.join(path, 'test_features.csv'))
    Y0 = pd.read_csv(os.path.join(path, 'train_targets_nonscored.csv'))
    Y = pd.read_csv(os.path.join(path, 'train_targets_scored.csv'))
    submission = pd.read_csv(os.path.join(path, 'sample_submission.csv'))
    
    keep_rows = train_features['cp_type']!='ctl_vehicle'
    train = train_features.copy()
    train = train[keep_rows].reset_index(drop=True)
    origin_Y = Y.copy()
    origin_Y = origin_Y.drop('sig_id', axis=1)
    Y0 = Y0[keep_rows].reset_index(drop=True) # nonscored pretrain
    Y0 = Y0.drop('sig_id', axis=1)
    Y = Y[keep_rows].reset_index(drop=True)
    Y = Y.drop('sig_id', axis=1)
    submission.iloc[:,1:] = 0

    # label smoothing
    Y_smooth = smooth_one_hot(Y, classes=2, smoothing=0.001)
    Y0_smooth = smooth_one_hot(Y0, classes=2, smoothing=0.001)
    return train, test_features, Y0, Y, Y0_smooth, Y_smooth, submission, origin_Y

In [4]:
def smooth_one_hot(Y, classes: int, smoothing=0.001):
    """
    Y_train : one-hot encoding dataframe
    if smoothing == 0, it's one-hot method
    if 0 < smoothing < 1, it's smooth method

    confidence = 1.0 - label_smoothing
    return y_true * confidence + (label_smoothing / num_classes)
    """
    assert 0 <= smoothing < 1
    Y_smooth = Y.copy()
    confidence = 1.0 - smoothing
    Y_smooth.replace(1, confidence + smoothing / classes, inplace=True)
    Y_smooth.replace(0, smoothing/classes, inplace=True)
    return Y_smooth

In [5]:
def apply_rankgauss(df1, df2, n_quantiles, ouput_distribution):
    GENES = [col for col in df1.columns if col.startswith('g-')]
    CELLS = [col for col in df1.columns if col.startswith('c-')]
    cols = GENES + CELLS
    
    qt = QuantileTransformer(n_quantiles=n_quantiles, random_state=0, output_distribution=ouput_distribution)
    df1[cols] = qt.fit_transform(df1[cols])
    df2[cols] = qt.transform(df2[cols])
    return df1, df2


def rank_gauss(train_features, test_features):
    GENES = [col for col in train_features.columns if col.startswith('g-')]
    CELLS = [col for col in train_features.columns if col.startswith('c-')]
    
    for col in (GENES + CELLS):
        transformer = QuantileTransformer(n_quantiles=100, random_state=0, output_distribution="normal")
        vec_len = len(train_features[col].values)
        vec_len_test = len(test_features[col].values)
        raw_vec = train_features[col].values.reshape(vec_len, 1)
        transformer.fit(raw_vec)

        train_features[col] = transformer.transform(raw_vec).reshape(1, vec_len)[0]
        test_features[col] = transformer.transform(test_features[col].values.reshape(vec_len_test, 1)).reshape(1, vec_len_test)[0]
    return train_features, test_features

In [6]:
def pre_process_0(train_data, test_data):
    GENES = [col for col in train_data.columns if col.startswith('g-')]
    CELLS = [col for col in train_data.columns if col.startswith('c-')]
    
    def fe_stats(train, test):
        for df in [train, test]:
            df['g_sum'] = df[GENES].sum(axis = 1)
            df['g_mean'] = df[GENES].mean(axis = 1)
            df['g_std'] = df[GENES].std(axis = 1)
            df['g_kurt'] = df[GENES].kurtosis(axis = 1)
            df['g_skew'] = df[GENES].skew(axis = 1)
            df['c_sum'] = df[CELLS].sum(axis = 1)
            df['c_mean'] = df[CELLS].mean(axis = 1)
            df['c_std'] = df[CELLS].std(axis = 1)
            df['c_kurt'] = df[CELLS].kurtosis(axis = 1)
            df['c_skew'] = df[CELLS].skew(axis = 1)
            df['gc_sum'] = df[GENES + CELLS].sum(axis = 1)
            df['gc_mean'] = df[GENES + CELLS].mean(axis = 1)
            df['gc_std'] = df[GENES + CELLS].std(axis = 1)
            df['gc_kurt'] = df[GENES + CELLS].kurtosis(axis = 1)
            df['gc_skew'] = df[GENES + CELLS].skew(axis = 1)
        return train, test

    def c_squared(train, test):
        for df in [train, test]:
            for feature in CELLS:
                df[f'squared_{feature}'] = df[feature] ** 2
        return train, test

    train_data, test_data = fe_stats(train_data, test_data)
    train_data, test_data = c_squared(train_data, test_data)

    return train_data, test_data

In [7]:
def pre_process(train_data, test_data):
    GENES = [col for col in train_data.columns if col.startswith('g-')]
    CELLS = [col for col in train_data.columns if col.startswith('c-')]
    
    # drop no use feature
    train_data = train_data.drop(['cp_type', 'sig_id'], axis=1)
    test_data = test_data.drop(['cp_type', 'sig_id'], axis=1)
    
    # get dummy, one hot encoding
    train_data = pd.get_dummies(train_data, columns=['cp_time', 'cp_dose'], drop_first=True)
    test_data = pd.get_dummies(test_data, columns=['cp_time', 'cp_dose'], drop_first=True)

    #train_data, test_data = rank_gauss(train_data, test_data)
    train_data, test_data = apply_rankgauss(train_data, test_data, 100, 'normal')
    
    # PCA from kernel https://www.kaggle.com/ragnar123/moa-dnn-feature-engineering
    # this improve CV a little but LB doesn't change
    def create_pca(train, test, colunm, n_components, kind='g'):
        # add pca to train data
        pca = PCA(n_components)
        PCA_data = pca.fit_transform(train[colunm])
        PCA_data = pd.DataFrame(PCA_data, columns=[f'PCA_{kind}-{i}' for i in range(PCA_data.shape[1])])
        train = pd.concat((train, PCA_data), axis=1)
        # train = train.drop(colunm, axis=1)
        
        # add pca to test data
        PCA_data = pca.transform(test[colunm])
        PCA_data = pd.DataFrame(PCA_data, columns=[f'PCA_{kind}-{i}' for i in range(PCA_data.shape[1])])
        test = pd.concat((test, PCA_data), axis=1)
        # test = test.drop(colunm, axis=1)
        return train, test

    train_data, test_data = create_pca(train_data, test_data, GENES, 45, kind='g')
    train_data, test_data = create_pca(train_data, test_data, CELLS, 15, kind='c')

    return train_data, test_data

In [8]:
class Model(nn.Module):
    def __init__(self, num_features, num_targets, hidden_size=512, init_bias=0):
        super(Model, self).__init__()
        
        self.batch_norm1 = nn.BatchNorm1d(num_features)
        self.dropout1 = nn.Dropout(0.2)
        self.dense1 = nn.Linear(num_features, hidden_size)

        self.batch_norm2 = nn.BatchNorm1d(hidden_size)
        self.dense2 = nn.Linear(hidden_size, hidden_size // 2)

        self.batch_norm3 = nn.BatchNorm1d(hidden_size // 2)

        self.dense3 = nn.Linear(hidden_size // 2, num_targets)
        self.dense3.bias.data = init_bias

    def forward(self, x):
        x = self.batch_norm1(x)
        x = self.dropout1(x)
        x = F.relu(self.dense1(x))
        
        x = self.batch_norm2(x)
        x = F.relu(self.dense2(x))

        x = self.batch_norm3(x)
        x = self.dense3(x)
        
        return x

In [9]:
class MoADataset:
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        dct = {
            'x': torch.tensor(self.features[idx, :], dtype=torch.float),
            'y': torch.tensor(self.targets[idx, :], dtype=torch.float)
        }
        return dct


class TestDataset:
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        dct = {
            'x': torch.tensor(self.features[idx, :], dtype=torch.float)
        }
        return dct

In [10]:
def train_fn(model, optimizer, scheduler, loss_fn, dataloader, device):
    model.train()
    final_loss = 0

    for data in dataloader:
        optimizer.zero_grad()
        inputs, targets = data['x'].to(device), data['y'].to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()

        final_loss += loss.item()

    final_loss /= len(dataloader)

    return final_loss

def BCELoss_with_clip(preds, targets, p_min=0.0001, p_max=0.9999):
    loss_fn = nn.BCELoss()
    loss = loss_fn(torch.clamp(preds, p_min, p_max), targets)
    return loss

def valid_fn(model, loss_fn, dataloader, device):
    model.eval()
    final_loss = 0
    valid_preds = []

    for data in dataloader:
        inputs, targets = data['x'].to(device), data['y'].to(device)
        outputs = model(inputs)
        
        loss = BCELoss_with_clip(outputs.sigmoid().detach(), targets)
        #loss = loss_fn(outputs, targets)

        final_loss += loss.item()
        valid_preds.append(outputs.sigmoid().detach().cpu().numpy())

    final_loss /= len(dataloader)
    valid_preds = np.concatenate(valid_preds)

    return final_loss, valid_preds


def inference_fn(model, dataloader, device):
    model.eval()
    preds = []

    for data in dataloader:
        inputs = data['x'].to(device)

        with torch.no_grad():
            outputs = model(inputs)

        preds.append(outputs.sigmoid().detach().cpu().numpy())
    preds = np.concatenate(preds)
    return preds

In [11]:
def load_pretrain(model):
    pretrained_dict = torch.load('model.pth')
    model_dict = model.state_dict()
    pretrained_dict =  {k: v for k, v in pretrained_dict.items() if k in model_dict} 
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)
    return model

In [12]:
def run_train(seed, X_train, Y_train, Y_train_smooth, x_test, train_index, valid_index):
    INIT_BIAS = torch.Tensor(np.log(Y_train.values.mean(axis=0)))
    seed_everything(seed)
    x_train, x_valid = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train, y_valid = Y_train_smooth.iloc[train_index], Y_train.iloc[valid_index]

    train_dataset = MoADataset(x_train.values, y_train.values)
    valid_dataset = MoADataset(x_valid.values, y_valid.values)
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    validloader = torch.utils.data.DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = Model(
        num_features=x_train.shape[1],
        num_targets=y_train.shape[1],
        hidden_size=HIDDEN_SIZE,
        init_bias=INIT_BIAS
    )

    model.to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer=optimizer, pct_start=0.05, div_factor=1e4,
                                              max_lr=0.01, epochs=EPOCHS, steps_per_epoch=len(trainloader))
    loss_fn = nn.BCEWithLogitsLoss()

    oof = np.zeros((X_train.shape[0], Y_train.shape[1]))
    best_loss = np.inf

    for epoch in range(EPOCHS):
        train_loss = train_fn(model, optimizer, scheduler, loss_fn, trainloader, DEVICE)
        valid_loss, valid_preds = valid_fn(model, loss_fn, validloader, DEVICE)
        if valid_loss < best_loss:
            best_loss = valid_loss
            oof[valid_index] = valid_preds
            torch.save(model.state_dict(), "model.pth")
            print(f"EPOCH: %d \t LR: %f \t train_loss: %f \t valid_loss: %f, New Best!" % (epoch, scheduler.get_lr()[0], train_loss, valid_loss))
        else:
            print(f"EPOCH: %d \t LR: %f \t train_loss: %f \t valid_loss: %f" % (epoch, scheduler.get_lr()[0], train_loss, valid_loss))

    # --------------------- PREDICTION---------------------
    testdataset = TestDataset(x_test.values)
    testloader = torch.utils.data.DataLoader(testdataset, batch_size=BATCH_SIZE, shuffle=False)

    model = Model(
        num_features=x_test.shape[1],
        num_targets=Y_train.shape[1],
        hidden_size=HIDDEN_SIZE,
        init_bias=INIT_BIAS
    )
    model.load_state_dict(torch.load("model.pth"))
    model.to(DEVICE)
    predictions = inference_fn(model, testloader, DEVICE)

    return oof, predictions

In [13]:
def create_folds(df_y, FOLDS, SEED):
    # LOCATE DRUGS
    vc = df_y.drug_id.value_counts()    
    vc1 = vc.loc[(vc==6)|(vc==12)|(vc==18)].index.sort_values()
    vc2 = vc.loc[(vc!=6)&(vc!=12)&(vc!=18)].index.sort_values()
    
    # TARGETS
    targets = [x for x in df_y.columns if x not in ['sig_id', 'drug_id']]

    # STRATIFY DRUGS 18X OR LESS
    dct1 = {}; dct2 = {}
    skf = MultilabelStratifiedKFold(n_splits=FOLDS, shuffle=True, 
              random_state=SEED)
    tmp = df_y.groupby('drug_id')[targets].mean().loc[vc1]
    for fold, (idxT, idxV) in enumerate(skf.split(tmp, tmp[targets])):
        dd = {k:fold for k in tmp.index[idxV].values}
        dct1.update(dd)

    # STRATIFY DRUGS MORE THAN 18X
    skf = MultilabelStratifiedKFold(n_splits=FOLDS, shuffle=True, 
              random_state=SEED)
    tmp = df_y.loc[df_y.drug_id.isin(vc2)].reset_index(drop=True)
    for fold, (idxT, idxV) in enumerate(skf.split(tmp, tmp[targets])):
        dd = {k:fold for k in tmp.sig_id[idxV].values}
        dct2.update(dd)

    # ASSIGN FOLDS
    df_y['Fold'] = df_y.drug_id.map(dct1)
    df_y.loc[df_y.Fold.isna(),'Fold'] = df_y.loc[df_y.Fold.isna(),'sig_id'].map(dct2)
    df_y.Fold = df_y.Fold.astype('int8')
    return df_y

In [14]:
def log_loss_numpy(y_pred, y_true):
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    loss = np.where(y_true == 1, -np.log(y_pred), -np.log(1 - y_pred))
    return loss.mean()

In [15]:
# NEW
def run_k_fold(seed, X_train, Y_train, Y_train_smooth, NFOLDS):
    oof = np.zeros((X_train.shape[0], Y_train.shape[1]))
    predictions = np.zeros((X_test.shape[0], Y_train.shape[1]))

    y_develop = pd.read_csv('../input/lish-moa/train_targets_scored.csv')
    x_develop = pd.read_csv('../input/lish-moa/train_features.csv')
    keep_rows = x_develop['cp_type']!='ctl_vehicle'
    y_develop = y_develop[keep_rows]
    
    drugs = pd.read_csv('../input/lish-moa/train_drug.csv')
    y_develop = y_develop.merge(drugs, how='left', on='sig_id')
    y_develop = create_folds(y_develop, NFOLDS, seed)

    for foldno in np.sort(y_develop['Fold'].unique()):
        train_index = y_develop[y_develop['Fold']!=foldno].index
        valid_index = y_develop[y_develop['Fold']==foldno].index
        print(f"\nFold-%d" % (foldno))
        print('Train Sample Size: %d, Validation Sample Size: %d' % (len(train_index), len(valid_index)))
        oof_, pred_ = run_train(seed, X_train, Y_train, Y_train_smooth, X_test, train_index, valid_index)

        predictions += pred_ / NFOLDS
        oof += oof_

    return oof, predictions

In [16]:
def generate_result_with_clip(oof, predictions, origin_Y, Y_train, ctl_idx, use_ctl=False, p_min=0.0001, p_max=0.9999):
    # 使用clip计算cv
    oof = np.clip(oof, p_min, p_max)
    if use_ctl:
        num1 = origin_Y.shape[0] - Y_train.shape[0]
        num2 = Y_train.shape[1]
        ctl_data = np.zeros((num1, num2))
        ctl_pd = pd.DataFrame(data=ctl_data, columns=Y_train.columns)
        Y_train = Y_train.append(ctl_pd, ignore_index=True)
        oof = np.vstack((oof, ctl_data))
    score = log_loss_numpy(oof, Y_train.values)
    print("CV log_loss: %f" % score)

    # generate submission
    # using clip
    predictions = np.clip(predictions, p_min, p_max)
    predictions[ctl_idx] = 0
    submission.iloc[:, 1:] = predictions
    submission.to_csv('submission.csv', index=False)

In [17]:
def optimize_weights(ps, labels):
    if isinstance(ps, list):
        ps = anp.stack(ps)
    
    weights = anp.random.dirichlet([2]*len(ps),size=1).reshape(len(ps)).tolist() + [1]
    L = labels.values

    def log_loss_numpy(y_pred, y_true=L):
        y_true = anp.array(y_true).ravel()
        y_pred = anp.array(y_pred).ravel()
        y_pred = anp.clip(y_pred, 1e-15, 1 - 1e-15)
        loss = anp.where(y_true == 1, -anp.log(y_pred), -anp.log(1 - y_pred))
        return loss.mean()


    def individual_log_loss(ps):
        for i, p  in enumerate(ps):
            print(f'Log Loss of M%d: %.7f' % (i, log_loss_numpy(p)))


    def calc_oof_blend(ws, ps):
        return anp.squeeze(anp.matmul(ws.reshape(1, 1, len(ws)), anp.transpose(ps, [1, 0, 2])))


    def Lagrange_func(params):
        ws = params[:-1]
        _lambda = params[-1]
        ws = anp.array(ws)
        oof_blend = calc_oof_blend(ws, ps)
        return log_loss_numpy(oof_blend) - _lambda * (ws.sum() - 1.)


    def Lagrange_obj(params):
        ws = params[:-1]
        grad_L = grad(Lagrange_func)
        pars = grad_L(params)
        dLdws = pars[:-1]
        # dldlam = pars[-1]
        res = anp.append(dLdws, sum(ws) - 1.)
        return res


    individual_log_loss(ps)
    start_time = time()
    pars = fsolve(Lagrange_obj, weights)
    ws = pars[:-1]
    time_elapsed = time() - start_time
    print(f'Optimized in %.2fs' % time_elapsed)
    print('Optimized Weights:', ws)
    oof_b = calc_oof_blend(ws, ps)
    optimized_cv = log_loss_numpy(oof_b)
    print('Optimised Blend OOF Score:', optimized_cv)
    return ws, optimized_cv

In [18]:
# hyper param
BATCH_SIZE = 128
DEVICE = ('cuda' if torch.cuda.is_available() else 'cpu')
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
HIDDEN_SIZE = 512
EPOCHS = 50
NFOLDS = 7
SEED = [14, 16, 77, 1984, 42]

In [19]:
X_train, X_test, Y_pretrain, Y_train, Y_pretrain_smooth, Y_train_smooth, submission, origin_Y = load_data()
ctl_idx = X_test[X_test['cp_type'] == 'ctl_vehicle'].index
X_train, X_test = pre_process_0(X_train, X_test)
X_train, X_test = pre_process(X_train, X_test)

In [20]:
oof = np.empty(shape=(len(SEED), Y_train.shape[0], Y_train.shape[1]))
predictions = np.empty(shape=(len(SEED), X_test.shape[0], Y_train.shape[1]))
start_time = time()
for i, seed in enumerate(SEED):
    print(f'\n\nSeed-%d (%d)' % (i, seed))
    oof_, predictions_ = run_k_fold(seed, X_train, Y_train, Y_train_smooth, NFOLDS)
    oof[i, :, :] = oof_
    predictions[i, :, :] = predictions_
    
np.save('Val_pred.npy', oof)
ws, optimized_cv = optimize_weights(oof, Y_train)
predictions = (predictions.transpose(1, 2, 0) * ws.reshape(-1, len(ws))).sum(axis=-1)
oof = (oof.transpose(1, 2, 0) * ws.reshape(-1, len(ws))).sum(axis=-1)

end_time = time()
print("total time consume:", end_time - start_time)



Seed-0 (14)

Fold-0
Train Sample Size: 18788, Validation Sample Size: 3160
EPOCH: 0 	 LR: 0.003472 	 train_loss: 0.023065 	 valid_loss: 0.018297, New Best!
EPOCH: 1 	 LR: 0.009065 	 train_loss: 0.020618 	 valid_loss: 0.018231, New Best!
EPOCH: 2 	 LR: 0.009997 	 train_loss: 0.020359 	 valid_loss: 0.018001, New Best!
EPOCH: 3 	 LR: 0.009975 	 train_loss: 0.020520 	 valid_loss: 0.017760, New Best!
EPOCH: 4 	 LR: 0.009931 	 train_loss: 0.020619 	 valid_loss: 0.018115
EPOCH: 5 	 LR: 0.009866 	 train_loss: 0.020770 	 valid_loss: 0.017980
EPOCH: 6 	 LR: 0.009780 	 train_loss: 0.020818 	 valid_loss: 0.017782
EPOCH: 7 	 LR: 0.009672 	 train_loss: 0.020816 	 valid_loss: 0.017724, New Best!
EPOCH: 8 	 LR: 0.009544 	 train_loss: 0.020757 	 valid_loss: 0.017928
EPOCH: 9 	 LR: 0.009396 	 train_loss: 0.020732 	 valid_loss: 0.017648, New Best!
EPOCH: 10 	 LR: 0.009229 	 train_loss: 0.020630 	 valid_loss: 0.017687
EPOCH: 11 	 LR: 0.009044 	 train_loss: 0.020528 	 valid_loss: 0.017679
EPOCH: 12 	 LR:

EPOCH: 9 	 LR: 0.009396 	 train_loss: 0.020558 	 valid_loss: 0.018575
EPOCH: 10 	 LR: 0.009229 	 train_loss: 0.020497 	 valid_loss: 0.018075, New Best!
EPOCH: 11 	 LR: 0.009044 	 train_loss: 0.020485 	 valid_loss: 0.018527
EPOCH: 12 	 LR: 0.008841 	 train_loss: 0.020519 	 valid_loss: 0.018226
EPOCH: 13 	 LR: 0.008621 	 train_loss: 0.020313 	 valid_loss: 0.017976, New Best!
EPOCH: 14 	 LR: 0.008385 	 train_loss: 0.020183 	 valid_loss: 0.018091
EPOCH: 15 	 LR: 0.008134 	 train_loss: 0.020137 	 valid_loss: 0.018409
EPOCH: 16 	 LR: 0.007870 	 train_loss: 0.020090 	 valid_loss: 0.018087
EPOCH: 17 	 LR: 0.007593 	 train_loss: 0.020047 	 valid_loss: 0.018040
EPOCH: 18 	 LR: 0.007305 	 train_loss: 0.019949 	 valid_loss: 0.018009
EPOCH: 19 	 LR: 0.007006 	 train_loss: 0.019855 	 valid_loss: 0.018140
EPOCH: 20 	 LR: 0.006699 	 train_loss: 0.019803 	 valid_loss: 0.017858, New Best!
EPOCH: 21 	 LR: 0.006385 	 train_loss: 0.019723 	 valid_loss: 0.017624, New Best!
EPOCH: 22 	 LR: 0.006064 	 train_l

EPOCH: 19 	 LR: 0.007006 	 train_loss: 0.020078 	 valid_loss: 0.017550, New Best!
EPOCH: 20 	 LR: 0.006699 	 train_loss: 0.019981 	 valid_loss: 0.017628
EPOCH: 21 	 LR: 0.006385 	 train_loss: 0.019849 	 valid_loss: 0.017533, New Best!
EPOCH: 22 	 LR: 0.006064 	 train_loss: 0.019821 	 valid_loss: 0.017662
EPOCH: 23 	 LR: 0.005739 	 train_loss: 0.019825 	 valid_loss: 0.017620
EPOCH: 24 	 LR: 0.005411 	 train_loss: 0.019635 	 valid_loss: 0.017457, New Best!
EPOCH: 25 	 LR: 0.005080 	 train_loss: 0.019601 	 valid_loss: 0.017412, New Best!
EPOCH: 26 	 LR: 0.004750 	 train_loss: 0.019530 	 valid_loss: 0.017299, New Best!
EPOCH: 27 	 LR: 0.004420 	 train_loss: 0.019434 	 valid_loss: 0.017351
EPOCH: 28 	 LR: 0.004093 	 train_loss: 0.019280 	 valid_loss: 0.017212, New Best!
EPOCH: 29 	 LR: 0.003770 	 train_loss: 0.019208 	 valid_loss: 0.017196, New Best!
EPOCH: 30 	 LR: 0.003453 	 train_loss: 0.019089 	 valid_loss: 0.017142, New Best!
EPOCH: 31 	 LR: 0.003142 	 train_loss: 0.018997 	 valid_loss

EPOCH: 28 	 LR: 0.004093 	 train_loss: 0.019248 	 valid_loss: 0.017286
EPOCH: 29 	 LR: 0.003770 	 train_loss: 0.019085 	 valid_loss: 0.017356
EPOCH: 30 	 LR: 0.003453 	 train_loss: 0.018962 	 valid_loss: 0.017543
EPOCH: 31 	 LR: 0.003142 	 train_loss: 0.018870 	 valid_loss: 0.017434
EPOCH: 32 	 LR: 0.002839 	 train_loss: 0.018663 	 valid_loss: 0.017337
EPOCH: 33 	 LR: 0.002546 	 train_loss: 0.018601 	 valid_loss: 0.017327
EPOCH: 34 	 LR: 0.002263 	 train_loss: 0.018419 	 valid_loss: 0.017217, New Best!
EPOCH: 35 	 LR: 0.001993 	 train_loss: 0.018295 	 valid_loss: 0.017378
EPOCH: 36 	 LR: 0.001735 	 train_loss: 0.018125 	 valid_loss: 0.017401
EPOCH: 37 	 LR: 0.001492 	 train_loss: 0.018029 	 valid_loss: 0.017321
EPOCH: 38 	 LR: 0.001264 	 train_loss: 0.017779 	 valid_loss: 0.017468
EPOCH: 39 	 LR: 0.001053 	 train_loss: 0.017612 	 valid_loss: 0.017473
EPOCH: 40 	 LR: 0.000859 	 train_loss: 0.017409 	 valid_loss: 0.017450
EPOCH: 41 	 LR: 0.000683 	 train_loss: 0.017252 	 valid_loss: 0.01

EPOCH: 38 	 LR: 0.001264 	 train_loss: 0.017708 	 valid_loss: 0.017534
EPOCH: 39 	 LR: 0.001053 	 train_loss: 0.017580 	 valid_loss: 0.017567
EPOCH: 40 	 LR: 0.000859 	 train_loss: 0.017315 	 valid_loss: 0.017543
EPOCH: 41 	 LR: 0.000683 	 train_loss: 0.017144 	 valid_loss: 0.017650
EPOCH: 42 	 LR: 0.000525 	 train_loss: 0.016945 	 valid_loss: 0.017587
EPOCH: 43 	 LR: 0.000388 	 train_loss: 0.016795 	 valid_loss: 0.017668
EPOCH: 44 	 LR: 0.000270 	 train_loss: 0.016636 	 valid_loss: 0.017697
EPOCH: 45 	 LR: 0.000173 	 train_loss: 0.016500 	 valid_loss: 0.017712
EPOCH: 46 	 LR: 0.000098 	 train_loss: 0.016393 	 valid_loss: 0.017738
EPOCH: 47 	 LR: 0.000043 	 train_loss: 0.016323 	 valid_loss: 0.017758
EPOCH: 48 	 LR: 0.000011 	 train_loss: 0.016262 	 valid_loss: 0.017718
EPOCH: 49 	 LR: 0.000000 	 train_loss: 0.016232 	 valid_loss: 0.017768

Fold-2
Train Sample Size: 18812, Validation Sample Size: 3136
EPOCH: 0 	 LR: 0.003472 	 train_loss: 0.023046 	 valid_loss: 0.018517, New Best!
EPOC

...

EPOCH: 6 	 LR: 0.009780 	 train_loss: 0.020769 	 valid_loss: 0.017887
EPOCH: 7 	 LR: 0.009672 	 train_loss: 0.020813 	 valid_loss: 0.018105
EPOCH: 8 	 LR: 0.009544 	 train_loss: 0.020745 	 valid_loss: 0.018004
EPOCH: 9 	 LR: 0.009396 	 train_loss: 0.020737 	 valid_loss: 0.017995
EPOCH: 10 	 LR: 0.009229 	 train_loss: 0.020669 	 valid_loss: 0.017651, New Best!
EPOCH: 11 	 LR: 0.009044 	 train_loss: 0.020600 	 valid_loss: 0.018030
EPOCH: 12 	 LR: 0.008841 	 train_loss: 0.020463 	 valid_loss: 0.017756
EPOCH: 13 	 LR: 0.008621 	 train_loss: 0.020487 	 valid_loss: 0.017678
EPOCH: 14 	 LR: 0.008385 	 train_loss: 0.020353 	 valid_loss: 0.017787
EPOCH: 15 	 LR: 0.008134 	 train_loss: 0.020305 	 valid_loss: 0.017642, New Best!
EPOCH: 16 	 LR: 0.007870 	 train_loss: 0.020169 	 valid_loss: 0.017472, New Best!
EPOCH: 17 	 LR: 0.007593 	 train_loss: 0.020159 	 valid_loss: 0.017660
EPOCH: 18 	 LR: 0.007305 	 train_loss: 0.020091 	 valid_loss: 0.017477
EPOCH: 19 	 LR: 0.007006 	 train_loss: 0.019912 

EPOCH: 16 	 LR: 0.007870 	 train_loss: 0.020201 	 valid_loss: 0.017550, New Best!
EPOCH: 17 	 LR: 0.007593 	 train_loss: 0.020708 	 valid_loss: 0.017846
EPOCH: 18 	 LR: 0.007305 	 train_loss: 0.020240 	 valid_loss: 0.017570
EPOCH: 19 	 LR: 0.007006 	 train_loss: 0.020192 	 valid_loss: 0.017579
EPOCH: 20 	 LR: 0.006699 	 train_loss: 0.020175 	 valid_loss: 0.017507, New Best!
EPOCH: 21 	 LR: 0.006385 	 train_loss: 0.020412 	 valid_loss: 0.017540
EPOCH: 22 	 LR: 0.006064 	 train_loss: 0.020076 	 valid_loss: 0.017649
EPOCH: 23 	 LR: 0.005739 	 train_loss: 0.019953 	 valid_loss: 0.017450, New Best!
EPOCH: 24 	 LR: 0.005411 	 train_loss: 0.019976 	 valid_loss: 0.017299, New Best!
EPOCH: 25 	 LR: 0.005080 	 train_loss: 0.019767 	 valid_loss: 0.017352
EPOCH: 26 	 LR: 0.004750 	 train_loss: 0.019633 	 valid_loss: 0.017345
EPOCH: 27 	 LR: 0.004420 	 train_loss: 0.019556 	 valid_loss: 0.017326
EPOCH: 28 	 LR: 0.004093 	 train_loss: 0.019550 	 valid_loss: 0.017345
EPOCH: 29 	 LR: 0.003770 	 train_

EPOCH: 25 	 LR: 0.005080 	 train_loss: 0.019752 	 valid_loss: 0.017320
EPOCH: 26 	 LR: 0.004750 	 train_loss: 0.019589 	 valid_loss: 0.017146
EPOCH: 27 	 LR: 0.004420 	 train_loss: 0.019420 	 valid_loss: 0.017026, New Best!
EPOCH: 28 	 LR: 0.004093 	 train_loss: 0.019341 	 valid_loss: 0.016991, New Best!
EPOCH: 29 	 LR: 0.003770 	 train_loss: 0.019488 	 valid_loss: 0.016963, New Best!
EPOCH: 30 	 LR: 0.003453 	 train_loss: 0.019181 	 valid_loss: 0.016974
EPOCH: 31 	 LR: 0.003142 	 train_loss: 0.019155 	 valid_loss: 0.016939, New Best!
EPOCH: 32 	 LR: 0.002839 	 train_loss: 0.018948 	 valid_loss: 0.016916, New Best!
EPOCH: 33 	 LR: 0.002546 	 train_loss: 0.018823 	 valid_loss: 0.017058
EPOCH: 34 	 LR: 0.002263 	 train_loss: 0.018643 	 valid_loss: 0.016838, New Best!
EPOCH: 35 	 LR: 0.001993 	 train_loss: 0.018535 	 valid_loss: 0.016992
EPOCH: 36 	 LR: 0.001735 	 train_loss: 0.018459 	 valid_loss: 0.016921
EPOCH: 37 	 LR: 0.001492 	 train_loss: 0.018182 	 valid_loss: 0.016818, New Best!


EPOCH: 35 	 LR: 0.001993 	 train_loss: 0.018239 	 valid_loss: 0.017384
EPOCH: 36 	 LR: 0.001735 	 train_loss: 0.018047 	 valid_loss: 0.017305
EPOCH: 37 	 LR: 0.001492 	 train_loss: 0.017887 	 valid_loss: 0.017288
EPOCH: 38 	 LR: 0.001264 	 train_loss: 0.017743 	 valid_loss: 0.017394
EPOCH: 39 	 LR: 0.001053 	 train_loss: 0.017565 	 valid_loss: 0.017312
EPOCH: 40 	 LR: 0.000859 	 train_loss: 0.017368 	 valid_loss: 0.017398
EPOCH: 41 	 LR: 0.000683 	 train_loss: 0.017219 	 valid_loss: 0.017394
EPOCH: 42 	 LR: 0.000525 	 train_loss: 0.016974 	 valid_loss: 0.017424
EPOCH: 43 	 LR: 0.000388 	 train_loss: 0.016781 	 valid_loss: 0.017488
EPOCH: 44 	 LR: 0.000270 	 train_loss: 0.016575 	 valid_loss: 0.017471
EPOCH: 45 	 LR: 0.000173 	 train_loss: 0.016450 	 valid_loss: 0.017506
EPOCH: 46 	 LR: 0.000098 	 train_loss: 0.016343 	 valid_loss: 0.017551
EPOCH: 47 	 LR: 0.000043 	 train_loss: 0.016273 	 valid_loss: 0.017487
EPOCH: 48 	 LR: 0.000011 	 train_loss: 0.016192 	 valid_loss: 0.017497
EPOCH:

EPOCH: 45 	 LR: 0.000173 	 train_loss: 0.016912 	 valid_loss: 0.016654
EPOCH: 46 	 LR: 0.000098 	 train_loss: 0.016718 	 valid_loss: 0.016717
EPOCH: 47 	 LR: 0.000043 	 train_loss: 0.016615 	 valid_loss: 0.016648
EPOCH: 48 	 LR: 0.000011 	 train_loss: 0.016590 	 valid_loss: 0.016609
EPOCH: 49 	 LR: 0.000000 	 train_loss: 0.016525 	 valid_loss: 0.016629

Fold-5
Train Sample Size: 18793, Validation Sample Size: 3155
EPOCH: 0 	 LR: 0.003472 	 train_loss: 0.022976 	 valid_loss: 0.018416, New Best!
EPOCH: 1 	 LR: 0.009065 	 train_loss: 0.020533 	 valid_loss: 0.018003, New Best!
EPOCH: 2 	 LR: 0.009997 	 train_loss: 0.020320 	 valid_loss: 0.018220
EPOCH: 3 	 LR: 0.009975 	 train_loss: 0.020522 	 valid_loss: 0.018309
EPOCH: 4 	 LR: 0.009931 	 train_loss: 0.020687 	 valid_loss: 0.018099
EPOCH: 5 	 LR: 0.009866 	 train_loss: 0.020732 	 valid_loss: 0.018009
EPOCH: 6 	 LR: 0.009780 	 train_loss: 0.020739 	 valid_loss: 0.018017
EPOCH: 7 	 LR: 0.009672 	 train_loss: 0.020763 	 valid_loss: 0.018147


In [21]:
generate_result_with_clip(oof, predictions, origin_Y, Y_train, ctl_idx, use_ctl=False)

CV log_loss: 0.016759
